# Agent Memory, Vector Stores & RAG

**WatSPEED Agentic AI prep — Week 3-4 - memory & RAG**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Agent memory: episodic, semantic, retrieval

The syllabus names *vector stores, episodic memory, and RAG*. They solve different problems:

- **Working memory** — the message list. Dies at the end of the run. (Notebook 04.)
- **Episodic memory** — what happened in past sessions. "Last week you asked about region."
- **Semantic memory / RAG** — facts fetched from a corpus at query time, so the model
  answers from *your* data instead of its training set.

RAG is three steps: **embed -> retrieve -> stuff into the prompt.** That's it.

In [2]:
# `embed` here is bag-of-words. Real embeddings are dense vectors from a trained model,
# but the retrieval mechanics below are identical - which is the point.
v = embed("Trust in AI is lower among younger respondents")
show("Sparse 'embedding' (first 5 dims)", dict(list(v.items())[:5]))

a = embed("younger respondents distrust AI")
b = embed("pandas groupby aggregates weights")
print(f"\nsimilar pair   : {cosine(v, a):.3f}")
print(f"unrelated pair : {cosine(v, b):.3f}")

Sparse 'embedding' (first 5 dims):
  {
    "trust": 0.35355339059327373,
    "in": 0.35355339059327373,
    "ai": 0.35355339059327373,
    "is": 0.35355339059327373,
    "lower": 0.35355339059327373
  }

similar pair   : 0.530
unrelated pair : 0.000


### Build a store and retrieve against it

In [3]:
store = VectorStore()
corpus = [
    ("Respondents aged 18-29 report a mean AI trust of 2.81 out of 5.", {"source": "codebook", "var": "age_group"}),
    ("Respondents aged 60+ report a mean AI trust of 3.44 out of 5.",   {"source": "codebook", "var": "age_group"}),
    ("Trust in AI rises with formal education level (p < 0.01).",       {"source": "findings", "var": "education"}),
    ("Survey weights must be applied before reporting any mean.",        {"source": "methods",  "var": "weight"}),
    ("Missing values are coded -9 and must be recoded to None.",         {"source": "methods",  "var": "missing"}),
]
for text, meta in corpus:
    store.add(text, **meta)

print(f"{len(store)} documents indexed\n")
for score, text, meta in store.search("how does age affect trust in AI?", k=3):
    print(f"  {score:.3f}  [{meta['source']:<8}] {text}")

5 documents indexed

  0.342  [findings] Trust in AI rises with formal education level (p < 0.01).
  0.189  [codebook] Respondents aged 60+ report a mean AI trust of 3.44 out of 5.
  0.183  [codebook] Respondents aged 18-29 report a mean AI trust of 2.81 out of 5.


### The 'A' in RAG — augmenting the prompt

Retrieval is only useful if the retrieved text reaches the model. This function is the
whole of RAG, and the `Cite the source` instruction is what makes the answer checkable.

In [4]:
def rag_prompt(store: VectorStore, question: str, k: int = 3) -> list[dict]:
    hits = store.search(question, k=k)
    context = "\n".join(f"[{i+1}] ({m['source']}) {t}" for i, (_, t, m) in enumerate(hits))
    return [
        {"role": "system", "content":
            "Answer ONLY from the context below. Cite the source number. "
            "If the context does not contain the answer, say so.\n\n" + context},
        {"role": "user", "content": question},
    ]

prompt = rag_prompt(store, "how does age affect trust in AI?")
print(prompt[0]["content"])

Answer ONLY from the context below. Cite the source number. If the context does not contain the answer, say so.

[1] (findings) Trust in AI rises with formal education level (p < 0.01).
[2] (codebook) Respondents aged 60+ report a mean AI trust of 3.44 out of 5.
[3] (codebook) Respondents aged 18-29 report a mean AI trust of 2.81 out of 5.


In [5]:
llm = get_llm([LLMResponse(content=
    "Trust rises with age: 2.81 for 18-29 vs 3.44 for 60+ [1][2].")])
print("\nAnswer:", llm.chat(prompt).content)

Using stub-llm (deterministic, offline) - no OPENAI_API_KEY found, so results are scripted.

Answer: Trust rises with age: 2.81 for 18-29 vs 3.44 for 60+ [1][2].


### The failure RAG is supposed to prevent

Ask something the corpus doesn't cover. A well-instructed model refuses. This is the
behaviour you must test for — silent fabrication is the whole risk of the pattern.

In [6]:
prompt = rag_prompt(store, "what is the median household income of respondents?")
banner("Retrieved context for an unanswerable question")
print(prompt[0]["content"])

llm = get_llm([LLMResponse(content=
    "The context does not contain household income, so I cannot answer that.")], verbose=False)
print("\nAnswer:", llm.chat(prompt).content)


Retrieved context for an unanswerable question
Answer ONLY from the context below. Cite the source number. If the context does not contain the answer, say so.

[1] (codebook) Respondents aged 60+ report a mean AI trust of 3.44 out of 5.
[2] (codebook) Respondents aged 18-29 report a mean AI trust of 2.81 out of 5.

Answer: The context does not contain household income, so I cannot answer that.


### Episodic memory

Persist facts *across* sessions, keyed by user. The store is the same; what changes is
that you write to it at the end of a run and read from it at the start of the next.

In [7]:
class EpisodicMemory:
    def __init__(self):
        self._store = VectorStore()

    def remember(self, user: str, fact: str, session: int) -> None:
        self._store.add(fact, user=user, session=session)

    def recall(self, user: str, cue: str, k: int = 2) -> list[str]:
        return [t for _, t, m in self._store.search(cue, k=k * 3) if m["user"] == user][:k]

mem = EpisodicMemory()
mem.remember("peter", "Peter works with Statistics Canada survey microdata.", session=1)
mem.remember("peter", "Peter prefers weighted estimates over raw counts.", session=1)
mem.remember("peter", "Peter is migrating analyses from SAS to Python.", session=2)
mem.remember("dana",  "Dana studies labour force participation.", session=1)

show("Recalled for peter", mem.recall("peter", "should I weight these estimates?"))
show("Recalled for dana",  mem.recall("dana", "should I weight these estimates?"))

Recalled for peter:
  [
    "Peter prefers weighted estimates over raw counts."
  ]
Recalled for dana:
  []
